# Bonus Task: Direct Preference Optimization (DPO)

This notebook adds a DPO stage on top of the Phase 2 SFT adapter without retraining `code_lora_has_aura.ipynb`.

Preference pairs are built as:
- `chosen`: successful ReAct traces reconstructed from the verified tool trajectories.
- `rejected`: corrupted traces that simulate hallucinated shortcuts, wrong tool logic, or malformed action inputs.

The notebook is training-only. It does not include inference code.


In [1]:
!pip uninstall -y bitsandbytes triton
!pip install --no-cache-dir --upgrade "bitsandbytes>=0.45.5" "accelerate" "transformers" "peft" "trl"


Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 188.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 144.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 263.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 136.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 190.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 198.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attemp

In [2]:
import torch
import bitsandbytes as bnb
import transformers
import trl
import peft

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("bitsandbytes:", bnb.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)


CUDA available: True
GPU: Tesla T4
Torch: 2.10.0+cu128
CUDA: 12.8
bitsandbytes: 0.49.2
transformers: 5.5.4
trl: 1.2.0
peft: 0.19.1


In [4]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


In [5]:
DRIVE_DIR = "/content/drive/MyDrive/CS_F425_Project"  # change if needed

# DPO-only controls.
FORCE_FRESH_DPO = False
SFT_SOURCE_OVERRIDE = None  # set this if you want to force a specific SFT adapter/checkpoint path

import gc
import glob as _glob
import inspect as _inspect
import json
import os
import random
import re
import sys
import time as _time
from numbers import Integral, Real

import pandas as pd
import torch
from datasets import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
)

sys.path.insert(0, DRIVE_DIR)
from tool_executor import ToolExecutor

MODEL_NAME_P2 = "mistralai/Mistral-7B-v0.1"
ADAPTER_DIR_P2 = f"{DRIVE_DIR}/mistral-react-adapter_divyam_duplicate"
CKPT_DIR_P2 = f"{DRIVE_DIR}/mistral-react-qlora_divyam_duplicate"
ADAPTER_DIR_DPO = f"{DRIVE_DIR}/mistral-react-dpo-adapter_divyam_duplicate3"
CKPT_DIR_DPO = f"{DRIVE_DIR}/mistral-react-dpo-ckpts_divyam_duplicate3"
MAX_PROMPT_LENGTH_DPO = 224
MAX_SEQ_LENGTH_DPO = 448
DPO_MAX_TRAIN_PAIRS = 384
DPO_MAX_VALID_PAIRS = 64
DPO_TRACE_MAX_OBS_CHARS = 96
DPO_FALLBACK_POOL_SIZE = 512

os.makedirs(ADAPTER_DIR_DPO, exist_ok=True)
os.makedirs(CKPT_DIR_DPO, exist_ok=True)

df = pd.read_csv(f"{DRIVE_DIR}/sales_data.csv")
with open(f"{DRIVE_DIR}/agent_trajectories_2k.json", "r", encoding="utf-8") as f:
    trajectories = json.load(f)

print("data shape:", df.shape)
print("trajectories:", len(trajectories))
print("SFT adapter dir:", ADAPTER_DIR_P2)
print("SFT checkpoints dir:", CKPT_DIR_P2)
print("DPO adapter dir:", ADAPTER_DIR_DPO)
print("DPO checkpoints dir:", CKPT_DIR_DPO)


def detect_training_state(adapter_dir, ckpt_dir):
    if os.path.isfile(f"{adapter_dir}/adapter_config.json"):
        return "TRAINING_COMPLETE", adapter_dir

    epoch_ckpts = _glob.glob(f"{ckpt_dir}/checkpoint-*")
    if epoch_ckpts:
        epoch_ckpts.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
        return "EPOCH_CHECKPOINT", epoch_ckpts[-1]

    timed_ckpts = _glob.glob(f"{adapter_dir}/timed_ckpt_step_*")
    if timed_ckpts:
        timed_ckpts.sort(key=lambda p: int(p.rsplit("_", 1)[-1]))
        return "TIMED_CHECKPOINT", timed_ckpts[-1]

    return "FRESH", None


def resolve_sft_source_path():
    candidates = []

    if SFT_SOURCE_OVERRIDE:
        candidates.append(SFT_SOURCE_OVERRIDE)

    if os.path.isdir(ADAPTER_DIR_P2):
        candidates.append(ADAPTER_DIR_P2)

    epoch_ckpts = _glob.glob(f"{CKPT_DIR_P2}/checkpoint-*")
    epoch_ckpts.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
    candidates.extend(reversed(epoch_ckpts))

    timed_ckpts = _glob.glob(f"{ADAPTER_DIR_P2}/timed_ckpt_step_*")
    timed_ckpts.sort(key=lambda p: int(p.rsplit("_", 1)[-1]))
    candidates.extend(reversed(timed_ckpts))

    for path in candidates:
        if path and os.path.isfile(f"{path}/adapter_config.json"):
            return path

    raise FileNotFoundError(
        "No SFT adapter/checkpoint with adapter_config.json was found. "
        "Run the SFT notebook once or set SFT_SOURCE_OVERRIDE to a valid adapter path."
    )


def _parse_numeric(val_str):
    return float(val_str) if "." in val_str else int(val_str)


def parse_agent_action(action_str):
    if action_str.startswith("filter_data"):
        m = re.search(r"column='([^']+)',\s*value=(-?[\d]+(?:\.\d+)?)", action_str)
        if m:
            return {
                "tool": "filter",
                "args": {"column": m.group(1), "op": "==", "value": _parse_numeric(m.group(2))},
            }
        m = re.search(r"column='([^']+)',\s*value='([^']+)'", action_str)
        if m:
            return {
                "tool": "filter",
                "args": {"column": m.group(1), "op": "==", "value": m.group(2)},
            }

    elif action_str.startswith("group_by"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "groupby", "args": {"column": m.group(1)}}

    elif action_str.startswith("aggregate_sum"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "sum"}}

    elif action_str.startswith("aggregate_mean"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "mean"}}

    elif action_str.startswith("aggregate_count"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"tool": "aggregate", "args": {"column": m.group(1), "agg": "count"}}

    elif action_str.startswith("sort_by"):
        m = re.search(r"column='([^']+)',\s*order='([^']+)'", action_str)
        if m:
            return {
                "tool": "sort",
                "args": {"column": m.group(1), "ascending": m.group(2) != "desc"},
            }

    elif action_str.startswith("top_k"):
        m = re.search(r"k=(\d+)", action_str)
        if m:
            return {"tool": "topk", "args": {"k": int(m.group(1))}}

    return None


def clean_scalar(value):
    if hasattr(value, "item"):
        try:
            value = value.item()
        except Exception:
            pass

    if isinstance(value, bool):
        return bool(value)
    if isinstance(value, Integral):
        return int(value)
    if isinstance(value, Real):
        return round(float(value), 4)
    return value


def result_to_python(actions, result_df):
    if result_df is None or (hasattr(result_df, "empty") and result_df.empty):
        return None
    if hasattr(result_df, "groups"):
        return None

    action_names = [a.split("(", 1)[0] for a in actions]

    if getattr(result_df, "shape", None) == (1, 1):
        return clean_scalar(result_df.iloc[0, 0])

    has_groupby = any(a.startswith("group_by") for a in action_names)
    has_sort_or_topk = any(a.startswith("sort_by") or a.startswith("top_k") for a in action_names)

    if isinstance(result_df, pd.DataFrame):
        if result_df.shape[1] == 2 and has_groupby and not has_sort_or_topk:
            key_col, value_col = result_df.columns
            return {str(row[key_col]): clean_scalar(row[value_col]) for _, row in result_df.iterrows()}

        return [
            {str(key): clean_scalar(value) for key, value in row.items()}
            for row in result_df.to_dict(orient="records")
        ]

    return clean_scalar(result_df)


def execute_action_sequence(actions, source_df):
    parsed = []
    for action in actions:
        parsed_action = parse_agent_action(action)
        if parsed_action is None:
            raise ValueError(f"Could not parse action: {action}")
        parsed.append(parsed_action)
    return ToolExecutor(source_df.copy()).execute(parsed)


def compute_answer(actions, source_df):
    if not actions:
        return None
    try:
        result = execute_action_sequence(actions, source_df)
        return result_to_python(actions, result)
    except Exception:
        return None


REACT_SYSTEM_PROMPT = """You are a data analysis agent. You have access to the following tools to analyze a sales dataset:

Function Descriptions:
[
  {"name": "filter_data", "description": "Filter rows where column equals value", "parameters": {"column": "str", "value": "str or int"}},
  {"name": "group_by", "description": "Group the data by a column", "parameters": {"column": "str"}},
  {"name": "aggregate_sum", "description": "Sum a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_mean", "description": "Average a numeric column", "parameters": {"column": "str"}},
  {"name": "aggregate_count", "description": "Count rows for a column", "parameters": {"column": "str"}},
  {"name": "sort_by", "description": "Sort by a column", "parameters": {"column": "str", "order": "asc or desc"}},
  {"name": "top_k", "description": "Select top k rows", "parameters": {"k": "int"}}
]

Schema: date (date), year (int), month (int), city (str), region (str), product (str), category (str), revenue (float), units_sold (int), cost (float), profit (float)

Use the Thought/Action/Action Input format. Wait for Observation after each action. End with Final Answer."""

REACT_PROMPT_TEMPLATE = """### System
{system}

### User Query
{question}

### Agent Scratchpad
{scratchpad}"""

DPO_SYSTEM_PROMPT = """You are a data analysis agent with tools: filter_data, group_by, aggregate_sum, aggregate_mean, aggregate_count, sort_by, top_k. Use Thought/Action/Action Input/Observation and end with Final Answer."""

DPO_PROMPT_TEMPLATE = """### System
{system}

### User Query
{question}

### Agent Scratchpad
"""

_THOUGHT_TEMPLATES = {
    "filter_data": "I need to filter the data by {args} to narrow down the dataset.",
    "group_by": "I should group the data by {args} to organize it.",
    "aggregate_sum": "I need to compute the sum of {args}.",
    "aggregate_mean": "I need to compute the average of {args}.",
    "aggregate_count": "I need to count the entries for {args}.",
    "sort_by": "I should sort the results by {args}.",
    "top_k": "I need to select the top {args} entries.",
}


def action_string_to_action_input(action_str):
    if action_str.startswith("filter_data"):
        m = re.search(r"column='([^']+)',\s*value=(-?[\d]+(?:\.\d+)?)", action_str)
        if m:
            return {"column": m.group(1), "value": _parse_numeric(m.group(2))}
        m = re.search(r"column='([^']+)',\s*value='([^']+)'", action_str)
        if m:
            return {"column": m.group(1), "value": m.group(2)}

    elif action_str.startswith("group_by"):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"column": m.group(1)}

    elif action_str.startswith(("aggregate_sum", "aggregate_mean", "aggregate_count")):
        m = re.search(r"column='([^']+)'", action_str)
        if m:
            return {"column": m.group(1)}

    elif action_str.startswith("sort_by"):
        m = re.search(r"column='([^']+)',\s*order='([^']+)'", action_str)
        if m:
            return {"column": m.group(1), "order": m.group(2)}

    elif action_str.startswith("top_k"):
        m = re.search(r"k=(\d+)", action_str)
        if m:
            return {"k": int(m.group(1))}

    raise ValueError(f"Unsupported action string: {action_str}")


def _quote_string(value):
    escaped = value.replace("\\", "\\\\").replace("'", "\\'")
    return f"'{escaped}'"


def _format_dsl_value(value):
    if isinstance(value, str):
        return _quote_string(value)
    if isinstance(value, bool):
        return "1" if value else "0"
    return str(value)


def action_input_to_action_string(action_name, action_input):
    if action_name == "filter_data":
        return (
            f"filter_data(column={_quote_string(str(action_input['column']))}, "
            f"value={_format_dsl_value(action_input['value'])})"
        )
    if action_name == "group_by":
        return f"group_by(column={_quote_string(str(action_input['column']))})"
    if action_name in {"aggregate_sum", "aggregate_mean", "aggregate_count"}:
        return f"{action_name}(column={_quote_string(str(action_input['column']))})"
    if action_name == "sort_by":
        return (
            f"sort_by(column={_quote_string(str(action_input['column']))}, "
            f"order={_quote_string(str(action_input['order']))})"
        )
    if action_name == "top_k":
        return f"top_k(k={int(action_input['k'])})"
    raise ValueError(f"Unknown action name: {action_name}")


def format_observation(result_df, max_chars=300):
    if result_df is None:
        return "No data returned."
    if hasattr(result_df, "groups") and not isinstance(result_df, pd.DataFrame):
        group_names = list(result_df.groups.keys())
        return f"Grouped result with {len(group_names)} groups. Sample groups: {[str(x) for x in group_names[:10]]}"
    if hasattr(result_df, "empty") and result_df.empty:
        return "Empty result."
    if getattr(result_df, "shape", None) == (1, 1):
        return str(result_df.iloc[0, 0])

    text = result_df.to_string(index=False) if isinstance(result_df, pd.DataFrame) else str(result_df)
    if len(text) > max_chars:
        text = text[:max_chars] + "..."
    return text


def build_trace_from_actions(actions, source_df, max_obs_chars=300):
    trace_lines = []
    executed_actions = []

    for idx, action_str in enumerate(actions):
        action_name = action_str.split("(", 1)[0]
        action_input = action_string_to_action_input(action_str)
        args_text = json.dumps(action_input, ensure_ascii=False)

        thought = _THOUGHT_TEMPLATES.get(action_name, "I need to use the next tool with {args}.").format(args=args_text)
        if idx == 0:
            thought = "Let me solve this step by step. " + thought

        trace_lines.append(f"Thought: {thought}")
        trace_lines.append(f"Action: {action_name}")
        trace_lines.append(f"Action Input: {args_text}")

        executed_actions.append(action_str)
        result = execute_action_sequence(executed_actions, source_df)
        trace_lines.append(f"Observation: {format_observation(result, max_chars=max_obs_chars)}")

    final_answer = compute_answer(actions, source_df)
    if final_answer is None:
        return None

    trace_lines.append("Thought: I now have all the information needed.")
    trace_lines.append(f"Final Answer: {json.dumps(final_answer, ensure_ascii=False)}")
    return "\n".join(trace_lines)


def build_react_trace(entry, source_df, max_obs_chars=300):
    try:
        return build_trace_from_actions(entry["actions"], source_df, max_obs_chars=max_obs_chars)
    except Exception:
        return None


def load_tokenizer_p2(padding_side="right"):
    tokenizer_p2 = AutoTokenizer.from_pretrained(MODEL_NAME_P2, trust_remote_code=True)
    if tokenizer_p2.pad_token is None:
        tokenizer_p2.pad_token = tokenizer_p2.eos_token
    tokenizer_p2.padding_side = padding_side
    return tokenizer_p2


data shape: (10000, 11)
trajectories: 2000
SFT adapter dir: /content/drive/MyDrive/CS_F425_Project/mistral-react-adapter_divyam_duplicate
SFT checkpoints dir: /content/drive/MyDrive/CS_F425_Project/mistral-react-qlora_divyam_duplicate
DPO adapter dir: /content/drive/MyDrive/CS_F425_Project/mistral-react-dpo-adapter_divyam_duplicate3
DPO checkpoints dir: /content/drive/MyDrive/CS_F425_Project/mistral-react-dpo-ckpts_divyam_duplicate3


In [6]:
VALID_GROUPBY_COLUMNS = ["city", "region", "product", "category", "year", "month"]
VALID_METRIC_COLUMNS = ["revenue", "units_sold", "cost", "profit"]
AGG_SWAP = {
    "aggregate_sum": "aggregate_mean",
    "aggregate_mean": "aggregate_sum",
    "aggregate_count": "aggregate_sum",
}


def canonical_answer(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True)


def choose_other(options, current):
    for option in options:
        if option != current:
            return option
    return current


def mutate_answer(value):
    if isinstance(value, bool):
        return not value
    if isinstance(value, int):
        return value + 7
    if isinstance(value, float):
        return round(value * 1.17 + 1.0, 4)
    if isinstance(value, str):
        return value + " (estimated)"
    if isinstance(value, list):
        if not value:
            return [{"hallucinated": True}]
        return list(reversed(value))
    if isinstance(value, dict):
        mutated = dict(value)
        first_key = next(iter(mutated), None)
        if first_key is None:
            return {"hallucinated": True}
        first_val = mutated[first_key]
        if isinstance(first_val, (int, float)) and not isinstance(first_val, bool):
            mutated[first_key] = mutate_answer(first_val)
        else:
            mutated[first_key] = "hallucinated"
        return mutated
    return {"hallucinated": True, "raw": str(value)}


def perturb_action(action_str):
    action_name = action_str.split("(", 1)[0]
    action_input = action_string_to_action_input(action_str)

    if action_name == "filter_data":
        value = action_input["value"]
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            action_input["value"] = type(value)(value + 1)
        else:
            action_input["value"] = str(value) + "_wrong"
        return action_input_to_action_string(action_name, action_input)

    if action_name == "group_by":
        action_input["column"] = choose_other(VALID_GROUPBY_COLUMNS, str(action_input["column"]))
        return action_input_to_action_string(action_name, action_input)

    if action_name in {"aggregate_sum", "aggregate_mean", "aggregate_count"}:
        swapped_action = AGG_SWAP[action_name]
        action_input["column"] = choose_other(VALID_METRIC_COLUMNS, str(action_input["column"]))
        return action_input_to_action_string(swapped_action, action_input)

    if action_name == "sort_by":
        action_input["order"] = "asc" if action_input["order"] == "desc" else "desc"
        return action_input_to_action_string(action_name, action_input)

    if action_name == "top_k":
        k = int(action_input["k"])
        action_input["k"] = max(1, k + 1 if k < 5 else k - 1)
        return action_input_to_action_string(action_name, action_input)

    return action_str


def build_hallucinated_rejected_trace(chosen_answer):
    wrong_answer = mutate_answer(chosen_answer)
    return "\n".join(
        [
            "Thought: I can answer this directly without checking the tools.",
            f"Final Answer: {json.dumps(wrong_answer, ensure_ascii=False)}",
        ]
    )


def build_bad_json_rejected_trace(chosen_answer):
    wrong_answer = mutate_answer(chosen_answer)
    return "\n".join(
        [
            "Thought: I can probably skip careful execution and still answer.",
            "Action: filter_data",
            "Action Input: {column: year, value: ???}",
            f"Final Answer: {json.dumps(wrong_answer, ensure_ascii=False)}",
        ]
    )

def build_dpo_prompt(question):
    prompt_template = globals().get("DPO_PROMPT_TEMPLATE")
    system_prompt = globals().get("DPO_SYSTEM_PROMPT")

    if prompt_template is not None and system_prompt is not None:
        return prompt_template.format(system=system_prompt, question=question)

    # Fallback for partial notebook reruns where the setup cell was not rerun.
    return REACT_PROMPT_TEMPLATE.format(
        system=REACT_SYSTEM_PROMPT,
        question=question,
        scratchpad="",
    )

def get_dpo_trace_max_obs_chars():
    return globals().get("DPO_TRACE_MAX_OBS_CHARS", 96)


def get_dpo_length_limits():
    return {
        "max_prompt_length": globals().get("MAX_PROMPT_LENGTH_DPO", 224),
        "max_seq_length": globals().get("MAX_SEQ_LENGTH_DPO", 448),
        "fallback_pool_size": globals().get("DPO_FALLBACK_POOL_SIZE", 512),
        "max_train_pairs": globals().get("DPO_MAX_TRAIN_PAIRS", 384),
        "max_valid_pairs": globals().get("DPO_MAX_VALID_PAIRS", 64),
    }


def build_preference_pair(entry, source_df):
    prompt = build_dpo_prompt(entry["query"])

    chosen_trace = build_react_trace(entry, source_df, max_obs_chars=get_dpo_trace_max_obs_chars())
    if chosen_trace is None:
        return None

    chosen_answer = compute_answer(entry["actions"], source_df)
    if chosen_answer is None:
        return None

    rejected_trace = None
    reject_type = None
    base_actions = list(entry["actions"])

    for idx in range(len(base_actions) - 1, -1, -1):
        corrupted_actions = list(base_actions)
        corrupted_actions[idx] = perturb_action(corrupted_actions[idx])
        if corrupted_actions[idx] == base_actions[idx]:
            continue

        try:
            candidate_trace = build_trace_from_actions(corrupted_actions, source_df, max_obs_chars=get_dpo_trace_max_obs_chars())
            candidate_answer = compute_answer(corrupted_actions, source_df)
        except Exception:
            candidate_trace = None
            candidate_answer = None

        if candidate_trace is None or candidate_answer is None:
            continue

        if canonical_answer(candidate_answer) == canonical_answer(chosen_answer):
            continue

        rejected_trace = candidate_trace
        reject_type = "corrupted_tool_logic"
        break

    if rejected_trace is None:
        if len(base_actions) % 2 == 0:
            rejected_trace = build_hallucinated_rejected_trace(chosen_answer)
            reject_type = "hallucinated_shortcut"
        else:
            rejected_trace = build_bad_json_rejected_trace(chosen_answer)
            reject_type = "malformed_action_input"

    return {
        "prompt": prompt,
        "chosen": chosen_trace + tokenizer_p2.eos_token,
        "rejected": rejected_trace + tokenizer_p2.eos_token,
        "reject_type": reject_type,
    }


def estimate_pair_lengths(row):
    prompt_len = len(tokenizer_p2(row["prompt"], add_special_tokens=False)["input_ids"])
    chosen_len = len(tokenizer_p2(row["chosen"], add_special_tokens=False)["input_ids"])
    rejected_len = len(tokenizer_p2(row["rejected"], add_special_tokens=False)["input_ids"])
    total_len = prompt_len + max(chosen_len, rejected_len)
    return {
        "prompt_len": prompt_len,
        "chosen_len": chosen_len,
        "rejected_len": rejected_len,
        "total_len": total_len,
    }


def pair_fits_length_limits(length_info):
    limits = get_dpo_length_limits()
    return (
        length_info["prompt_len"] <= limits["max_prompt_length"]
        and length_info["total_len"] <= limits["max_seq_length"]
    )


def summarize_lengths(length_infos, key):
    values = sorted(info[key] for info in length_infos)
    if not values:
        return None, None, None
    mid = values[len(values) // 2]
    return values[0], mid, values[-1]


tokenizer_p2 = load_tokenizer_p2(padding_side="right")

preference_rows = []
for entry in tqdm(trajectories, desc="Building DPO pairs"):
    row = build_preference_pair(entry, df)
    if row is not None:
        preference_rows.append(row)

print("Pairs before length filter:", len(preference_rows))
length_rows = [(row, estimate_pair_lengths(row)) for row in preference_rows]

prompt_stats = summarize_lengths([info for _, info in length_rows], "prompt_len")
total_stats = summarize_lengths([info for _, info in length_rows], "total_len")
print("Prompt len min/median/max:", prompt_stats)
print("Total  len min/median/max:", total_stats)

strict_rows = [row for row, info in length_rows if pair_fits_length_limits(info)]
print("Pairs after length filter :", len(strict_rows))

if strict_rows:
    preference_rows = strict_rows
else:
    print("No pairs fit the strict token budget. Falling back to the shortest pairs and relying on DPO truncation.")
    length_rows.sort(key=lambda item: (item[1]["prompt_len"], item[1]["total_len"]))
    limits = get_dpo_length_limits()
    fallback_keep = min(
        len(length_rows),
        max(limits["fallback_pool_size"], limits["max_train_pairs"] + limits["max_valid_pairs"]),
    )
    preference_rows = [row for row, _ in length_rows[:fallback_keep]]
    print("Fallback pairs retained   :", len(preference_rows))

random.seed(42)
random.shuffle(preference_rows)

limits = get_dpo_length_limits()
train_size = min(limits["max_train_pairs"], max(1, int(len(preference_rows) * 0.9)))
valid_cap = max(0, len(preference_rows) - train_size)
valid_size = min(limits["max_valid_pairs"], valid_cap)

raw_train_dpo = preference_rows[:train_size]
raw_valid_dpo = preference_rows[train_size : train_size + valid_size]

train_rows = [{k: row[k] for k in ("prompt", "chosen", "rejected")} for row in raw_train_dpo]
valid_rows = [{k: row[k] for k in ("prompt", "chosen", "rejected")} for row in raw_valid_dpo]

dpo_train_ds = Dataset.from_list(train_rows)
dpo_valid_ds = Dataset.from_list(valid_rows) if valid_rows else None

reject_hist = {}
for row in raw_train_dpo + raw_valid_dpo:
    reject_hist[row["reject_type"]] = reject_hist.get(row["reject_type"], 0) + 1

print("DPO train pairs:", len(dpo_train_ds))
print("DPO valid pairs:", 0 if dpo_valid_ds is None else len(dpo_valid_ds))
print("Reject type counts:", reject_hist)
print("\nSample prompt:\n")
print(train_rows[0]["prompt"][:700])
print("\nSample chosen completion:\n")
print(train_rows[0]["chosen"][:900])
print("\nSample rejected completion:\n")
print(train_rows[0]["rejected"][:900])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Building DPO pairs:   0%|          | 0/2000 [00:00<?, ?it/s]

Pairs before length filter: 2000
Prompt len min/median/max: (76, 82, 90)
Total  len min/median/max: (256, 332, 695)
Pairs after length filter : 1349
DPO train pairs: 384
DPO valid pairs: 64
Reject type counts: {'corrupted_tool_logic': 448}

Sample prompt:

### System
You are a data analysis agent with tools: filter_data, group_by, aggregate_sum, aggregate_mean, aggregate_count, sort_by, top_k. Use Thought/Action/Action Input/Observation and end with Final Answer.

### User Query
Average units_sold by city

### Agent Scratchpad


Sample chosen completion:

Thought: Let me solve this step by step. I should group the data by {"column": "city"} to organize it.
Action: group_by
Action Input: {"column": "city"}
Observation: Grouped result with 5 groups. Sample groups: ['Bangalore', 'Chennai', 'Delhi', 'Kolkata', 'Mumbai']
Thought: I need to compute the average of {"column": "units_sold"}.
Action: aggregate_mean
Action Input: {"column": "units_sold"}
Observation:      city  units_sold
Bangalo

## DPO Training

The cell below loads the latest available SFT adapter in trainable mode, then runs `DPOTrainer` with explicit `prompt` / `chosen` / `rejected` pairs as documented in TRL.

Reference:
- https://huggingface.co/docs/trl/en/dpo_trainer


In [ ]:
from peft import PeftModel, prepare_model_for_kbit_training
from trl import DPOConfig, DPOTrainer

# Reduce fragmentation before loading the trainable DPO model.
gc.collect()
torch.cuda.empty_cache()

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config_p2 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

SFT_SOURCE_PATH = resolve_sft_source_path()
DPO_STATE, DPO_CKPT_PATH = detect_training_state(ADAPTER_DIR_DPO, CKPT_DIR_DPO)
DPO_SKIP_TRAINING = (not FORCE_FRESH_DPO) and DPO_STATE in ("TRAINING_COMPLETE", "TIMED_CHECKPOINT")

print("Resolved SFT source:", SFT_SOURCE_PATH)
print("DPO state:", DPO_STATE)
print("DPO checkpoint:", DPO_CKPT_PATH)
print("Skip DPO training:", DPO_SKIP_TRAINING)

if not DPO_SKIP_TRAINING:
    dpo_resume_ckpt = None
    dpo_timed_resume_ckpt = None

    if not FORCE_FRESH_DPO:
        ckpts = _glob.glob(f"{CKPT_DIR_DPO}/checkpoint-*")
        if ckpts:
            ckpts.sort(key=lambda p: int(p.rsplit("-", 1)[-1]))
            dpo_resume_ckpt = ckpts[-1]
        else:
            timed_ckpts = _glob.glob(f"{ADAPTER_DIR_DPO}/timed_ckpt_step_*")
            if timed_ckpts:
                timed_ckpts.sort(key=lambda p: int(p.rsplit("_", 1)[-1]))
                dpo_timed_resume_ckpt = timed_ckpts[-1]

    base_model_dpo = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME_P2,
        quantization_config=bnb_config_p2,
        device_map="auto",
        trust_remote_code=True,
    )
    base_model_dpo.config.use_cache = False
    base_model_dpo = prepare_model_for_kbit_training(base_model_dpo, use_gradient_checkpointing=True)

    if dpo_timed_resume_ckpt is not None:
        print("Resuming DPO from timed adapter:", dpo_timed_resume_ckpt)
        model_dpo = PeftModel.from_pretrained(base_model_dpo, dpo_timed_resume_ckpt, is_trainable=True)
    else:
        print("Loading SFT adapter for DPO init:", SFT_SOURCE_PATH)
        model_dpo = PeftModel.from_pretrained(base_model_dpo, SFT_SOURCE_PATH, is_trainable=True)

    model_dpo.print_trainable_parameters()

    tokenizer_dpo = load_tokenizer_p2(padding_side="left")

    dpo_cfg_sig = set(_inspect.signature(DPOConfig.__init__).parameters.keys())
    dpo_trainer_sig = set(_inspect.signature(DPOTrainer.__init__).parameters.keys())

    eval_key = "eval_strategy" if "eval_strategy" in dpo_cfg_sig else "evaluation_strategy"
    tok_key = "processing_class" if "processing_class" in dpo_trainer_sig else "tokenizer"

    do_eval = dpo_valid_ds is not None and len(dpo_valid_ds) > 0
    batch_size = 1
    grad_accum = 16
    save_eval_steps = 25
    steps_per_epoch = max(1, len(dpo_train_ds) // (batch_size * grad_accum))
    warmup_steps = max(1, int(0.05 * steps_per_epoch * 2))

    dpo_kwargs = {
        "output_dir": CKPT_DIR_DPO,
        "seed": 42,
        "num_train_epochs": 2,
        "per_device_train_batch_size": batch_size,
        "gradient_accumulation_steps": grad_accum,
        "learning_rate": 5e-6,
        "lr_scheduler_type": "cosine",
        "warmup_steps": warmup_steps,
        "optim": "paged_adamw_8bit",
        "bf16": torch.cuda.is_bf16_supported(),
        "fp16": not torch.cuda.is_bf16_supported(),
        "logging_steps": 10,
        "save_strategy": "steps",
        "save_steps": save_eval_steps,
        "save_total_limit": 4,
        "report_to": "none",
        "beta": 0.1,
        "max_prompt_length": MAX_PROMPT_LENGTH_DPO,
        "max_length": MAX_SEQ_LENGTH_DPO,
        "precompute_ref_log_probs": False,
        "remove_unused_columns": False,
    }

    if "truncation_mode" in dpo_cfg_sig:
        dpo_kwargs["truncation_mode"] = "keep_end"

    if do_eval:
        dpo_kwargs[eval_key] = "steps"
        dpo_kwargs["eval_steps"] = save_eval_steps
        dpo_kwargs["load_best_model_at_end"] = True
        dpo_kwargs["metric_for_best_model"] = "eval_loss"
    else:
        dpo_kwargs[eval_key] = "no"

    dpo_args = DPOConfig(**{k: v for k, v in dpo_kwargs.items() if k in dpo_cfg_sig})

    class TimedCheckpointCallbackDPO(TrainerCallback):
        def __init__(self, adapter_dir, tokenizer, interval_min=5):
            self.adapter_dir = adapter_dir
            self.tokenizer = tokenizer
            self.interval_sec = interval_min * 60
            self.last_save = _time.time()

        def on_step_end(self, args, state, control, model=None, **kwargs):
            elapsed = _time.time() - self.last_save
            if elapsed >= self.interval_sec and model is not None:
                save_path = f"{self.adapter_dir}/timed_ckpt_step_{state.global_step}"
                os.makedirs(save_path, exist_ok=True)
                model.save_pretrained(save_path)
                self.tokenizer.save_pretrained(save_path)
                self.last_save = _time.time()
                print(
                    f"\n[DPO TimedCheckpoint] Saved adapter at step {state.global_step} "
                    f"-> {save_path} ({elapsed / 60:.1f} min)"
                )

    trainer_kwargs = {
        "model": model_dpo,
        tok_key: tokenizer_dpo,
        "args": dpo_args,
        "train_dataset": dpo_train_ds,
        "ref_model": None,
        "callbacks": [TimedCheckpointCallbackDPO(ADAPTER_DIR_DPO, tokenizer_dpo, interval_min=5)],
    }
    if do_eval:
        trainer_kwargs["eval_dataset"] = dpo_valid_ds

    trainer_dpo = DPOTrainer(**trainer_kwargs)

    print("Starting DPO training")
    print("train pairs:", len(dpo_train_ds))
    print("valid pairs:", 0 if dpo_valid_ds is None else len(dpo_valid_ds))
    print("max prompt len:", MAX_PROMPT_LENGTH_DPO)
    print("max seq len:", MAX_SEQ_LENGTH_DPO)
    total_steps = steps_per_epoch * 2

    print("batch size:", batch_size)
    print("grad accum:", grad_accum)
    print("steps/epoch:", steps_per_epoch)
    print("total steps:", total_steps)
    print("trainer checkpoints dir:", CKPT_DIR_DPO)
    print("timed adapter checkpoints dir:", ADAPTER_DIR_DPO)

    trainer_dpo.train(resume_from_checkpoint=dpo_resume_ckpt)

    model_dpo.save_pretrained(ADAPTER_DIR_DPO)
    tokenizer_dpo.save_pretrained(ADAPTER_DIR_DPO)
    print("saved final DPO adapter to:", ADAPTER_DIR_DPO)
else:
    print("DPO adapter already available at:", DPO_CKPT_PATH)
    print("Set FORCE_FRESH_DPO = True if you want to retrain DPO from the SFT adapter.")


Resolved SFT source: /content/drive/MyDrive/CS_F425_Project/mistral-react-qlora_divyam_duplicate/checkpoint-25
DPO state: FRESH
DPO checkpoint: None
Skip DPO training: False


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

Tokenizing eval dataset:   0%|          | 0/64 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting DPO training
train pairs: 384
valid pairs: 64
max prompt len: 224
max seq len: 448
batch size: 1
grad accum: 16
steps/epoch: 24
total steps: 48
trainer checkpoints dir: /content/drive/MyDrive/CS_F425_Project/mistral-react-dpo-ckpts_divyam_duplicate3
timed adapter checkpoints dir: /content/drive/MyDrive/CS_F425_Project/mistral-react-dpo-adapter_divyam_duplicate3

[DPO TimedCheckpoint] Saved adapter at step 1 -> /content/drive/MyDrive/CS_F425_Project/mistral-react-dpo-adapter_divyam_duplicate3/timed_ckpt_step_1 (5.3 min)


Step,Training Loss,Validation Loss
25,0.000284,0.000063



[DPO TimedCheckpoint] Saved adapter at step 2 -> /content/drive/MyDrive/CS_F425_Project/mistral-react-dpo-adapter_divyam_duplicate3/timed_ckpt_step_2 (5.0 min)

[DPO TimedCheckpoint] Saved adapter at step 4 -> /content/drive/MyDrive/CS_F425_Project/mistral-react-dpo-adapter_divyam_duplicate3/timed_ckpt_step_4 (10.0 min)

[DPO TimedCheckpoint] Saved adapter at step 5 -> /content/drive/MyDrive/CS_F425_Project/mistral-react-dpo-adapter_divyam_duplicate3/timed_ckpt_step_5 (5.4 min)

[DPO TimedCheckpoint] Saved adapter at step 6 -> /content/drive/MyDrive/CS_F425_Project/mistral-react-dpo-adapter_divyam_duplicate3/timed_ckpt_step_6 (5.2 min)

[DPO TimedCheckpoint] Saved adapter at step 8 -> /content/drive/MyDrive/CS_F425_Project/mistral-react-dpo-adapter_divyam_duplicate3/timed_ckpt_step_8 (10.1 min)

[DPO TimedCheckpoint] Saved adapter at step 9 -> /content/drive/MyDrive/CS_F425_Project/mistral-react-dpo-adapter_divyam_duplicate3/timed_ckpt_step_9 (5.1 min)

[DPO TimedCheckpoint] Saved ada